In [ ]:
"""
Week 1 - Day 1
MDP Design + Gym Environment
==============================

Building custom OpenAI Gymnasium
environment for dynamic pricing.

Internship Spec:
"build a custom Python environment
subclassing gym.Env"

Infotact DS/ML Internship — Project 2
"""

import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------
# Add src folder to Python path
# -------------------------------------

project_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
src_path = os.path.join(project_root, "src")

if src_path not in sys.path:
    sys.path.insert(0, src_path)

# -------------------------------------
# Import Environment
# -------------------------------------

from environment.pricing_env import (
    DynamicPricingEnv,
    PRICE_LEVELS,
)

plt.style.use("seaborn-v0_8")

print("✅ Environment imported successfully!")
print(f"Project Root : {project_root}")
print(f"Source Path  : {src_path}")
print(f"Price Levels : {PRICE_LEVELS}")

In [ ]:
# Create environment
env = DynamicPricingEnv(
    max_inventory=50,
    max_days=30
)

print("=== ENVIRONMENT SPECIFICATIONS ===\n")
print(f"Max Inventory    : {env.max_inventory}")
print(f"Max Days         : {env.max_days}")
print(f"Price Levels     : {env.price_levels}")
print(f"Action Space     : {env.action_space}")
print(f"Observation Space: {env.observation_space}")
print(f"\nState Space Size : "
      f"{env.max_inventory+1} × {env.max_days+1}"
      f" = {(env.max_inventory+1)*(env.max_days+1)}")

In [ ]:
# Visualize demand probability
# across prices and days

prices   = PRICE_LEVELS
days     = [1, 5, 10, 15, 20, 25, 30]

fig, ax = plt.subplots(figsize=(12, 6))

for day in days:
    probs = [
        env._get_demand_probability(p, day)
        for p in prices
    ]
    ax.plot(
        prices, probs,
        marker='o',
        linewidth=2,
        label=f'{day} days left'
    )

ax.set_title(
    'Demand Probability vs Price\n'
    'by Days Remaining',
    fontweight='bold',
    fontsize=13
)
ax.set_xlabel('Price ($)')
ax.set_ylabel('Purchase Probability')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
import os

os.makedirs("results", exist_ok=True)
plt.savefig("results/demand_function.png")
plt.show()
print("✅ Demand function plot saved!")

In [ ]:
# Run one episode with random agent
env.reset(seed=42)
obs, info = env.reset(seed=42)

history = []
total_reward = 0

print("Running random agent episode...\n")
while True:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = (
        env.step(action)
    )
    total_reward += reward
    history.append({
        'day'       : env.max_days - info['days_left'],
        'price'     : info['price'],
        'bought'    : info['bought'],
        'inventory' : info['inventory'],
        'revenue'   : info['total_revenue']
    })

    if terminated or truncated:
        break

history_df = pd.DataFrame(history)
print(f"Episode Complete!")
print(f"  Steps         : {len(history_df)}")
print(f"  Total Revenue : ${total_reward:.0f}")
print(f"  Items Sold    : "
      f"{50 - history_df['inventory'].iloc[-1]}")
print(f"\n{history_df.head(10)}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price trajectory
axes[0,0].plot(
    history_df['day'],
    history_df['price'],
    color='steelblue',
    linewidth=2,
    marker='o',
    markersize=4
)
axes[0,0].set_title('Price Trajectory',
                     fontweight='bold')
axes[0,0].set_xlabel('Day')
axes[0,0].set_ylabel('Price ($)')
axes[0,0].grid(True, alpha=0.3)

# Inventory over time
axes[0,1].plot(
    history_df['day'],
    history_df['inventory'],
    color='coral',
    linewidth=2
)
axes[0,1].set_title('Inventory Over Time',
                     fontweight='bold')
axes[0,1].set_xlabel('Day')
axes[0,1].set_ylabel('Remaining Inventory')
axes[0,1].grid(True, alpha=0.3)

# Cumulative revenue
axes[1,0].plot(
    history_df['day'],
    history_df['revenue'],
    color='green',
    linewidth=2
)
axes[1,0].set_title('Cumulative Revenue',
                     fontweight='bold')
axes[1,0].set_xlabel('Day')
axes[1,0].set_ylabel('Total Revenue ($)')
axes[1,0].grid(True, alpha=0.3)

# Sales per day
sales = history_df['bought'].astype(int)
axes[1,1].bar(
    history_df['day'],
    sales,
    color='gold',
    edgecolor='black'
)
axes[1,1].set_title('Sales per Day',
                     fontweight='bold')
axes[1,1].set_xlabel('Day')
axes[1,1].set_ylabel('Sold (0/1)')
axes[1,1].grid(True, alpha=0.3)

plt.suptitle(
    'Random Agent Episode Analysis\n'
    'Dynamic Pricing Environment',
    fontsize=14,
    fontweight='bold'
)
plt.tight_layout()
plt.savefig('results/random_episode.png')
plt.show()
print("✅ Episode plots saved!")

In [ ]:
print("╔══════════════════════════════════════════╗")
print("║    WEEK 1 DAY 1 — COMPLETE SUMMARY       ║")
print("╠══════════════════════════════════════════╣")
print("║  MDP DESIGN:                             ║")
print("║  State  : (inventory, days_left)         ║")
print("║  Actions: 6 price levels ($50-$300)      ║")
print("║  Reward : Revenue earned per sale        ║")
print("║  Penalty: -10 per unsold ticket          ║")
print("╠══════════════════════════════════════════╣")
print("║  ENVIRONMENT:                            ║")
print(f"║  Max Inventory : 50 tickets              ║")
print(f"║  Max Days      : 30 days                 ║")
print(f"║  State Space   : 1,581 states            ║")
print(f"║  Action Space  : 6 discrete prices       ║")
print("╠══════════════════════════════════════════╣")
print("║  DEMAND FUNCTION:                        ║")
print("║  ✅ Price sensitive (higher=less demand) ║")
print("║  ✅ Time sensitive (less days=less demand)║")
print("║  ✅ Stochastic noise added               ║")
print("╠══════════════════════════════════════════╣")
print("║  Tomorrow → Baseline Agents 🎯           ║")
print("╚══════════════════════════════════════════╝")